# Convert the trained figure model for Coral

This notebook uses **your `best.pt`**, not stock COCO weights. It converts to
full INT8 at 320x320, compiles with compiler 16, and packages the two-class
model for the C++ `--yolo-profile figure` option. No retraining is performed.

Use **Linux x86-64, Python 3.12** in Colab. CPU is sufficient; your current
Python 3.12 training session also works. If Colab's default Python differs,
choose a runtime version offering 3.12. All installs/conversion run in a
separate environment/process, avoiding the stale NumPy problem in notebooks.

Run cells in order. Have **figure_training_result.zip** and the original
**tabletop_dataset.zip** available. If both are already in `/content`, the
notebook reuses them; otherwise the upload cell asks for both.

Quantization uses only the **60 training images**. The 18 validation images
compare the PyTorch and full-INT8 models. The dataset is small and from one
recording; these scores cannot establish robustness or Coral latency.
Read the compiler report: some operations can remain on the CPU.

In [ ]:
import os, sys, platform, subprocess, json
from pathlib import Path
if platform.system() != "Linux" or platform.machine() != "x86_64" or sys.version_info[:2] != (3, 12):
    raise RuntimeError(f"Use a Colab Linux x86-64 runtime with Python 3.12; found {platform.platform()} / {sys.version}")
TOOLS = Path("/content/figure_export_tools")
TOOLS.mkdir(exist_ok=True)

SOURCES = {'export_figure.py': '#!/usr/bin/env python3\n"""Convert the reviewed figure checkpoint to INT8 and compile for Coral on Linux x86-64.\n\nThe C++ application performs all live inference, decoding, NMS, and fusion.\nThis offline tool verifies the checkpoint/dataset, calibrates using TRAIN only,\ncompares PyTorch and uncompiled INT8 on VAL, and records the real compiler report.\nIt does not train the model or claim a hardware benchmark.\n"""\nfrom pathlib import Path, PurePosixPath\nimport argparse\nimport hashlib\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport zipfile\n\nfrom export_yolo import ensure_compiler, COMPILER_SHA256\n\nNAMES = {0: "figure", 1: "stop_sign"}\nSTEM = "figure_yolov8n_320"\n\n\ndef sha(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef extract(archive, destination):\n    """Validate every ZIP member before extracting into a new, dedicated directory."""\n    destination = destination.resolve()\n    if destination.exists():\n        raise ValueError(f"Extraction directory already exists: {destination}; use a fresh output folder")\n    with zipfile.ZipFile(archive) as source:\n        members = source.infolist()\n        if sum(item.file_size for item in members) > 2_000_000_000:\n            raise ValueError("ZIP exceeds the 2 GB extraction limit")\n        seen = set()\n        for item in members:\n            name = PurePosixPath(item.filename)\n            if name.is_absolute() or ".." in name.parts or "\\\\" in item.filename or ":" in item.filename:\n                raise ValueError("Unsafe ZIP member path")\n            if (item.external_attr >> 16) & 0o170000 == 0o120000:\n                raise ValueError("ZIP symlinks are not supported")\n            if name in seen or not (destination / str(name)).resolve().is_relative_to(destination):\n                raise ValueError("Duplicate or escaping ZIP member")\n            seen.add(name)\n        source.extractall(destination)\n\n\ndef inspect_model(path, compiled=False):\n    """Inspect the FlatBuffer directly: the Colab machine has no USB TPU attached."""\n    from tensorflow.lite.python import schema_py_generated as schema\n    data = path.read_bytes()\n    if data[4:8] != b"TFL3":\n        raise ValueError("Not a TFLite FlatBuffer")\n    model = schema.Model.GetRootAsModel(data, 0)\n    graph = model.Subgraphs(0)\n    if graph.InputsLength() != 1 or graph.OutputsLength() != 1:\n        raise ValueError("Expected one input/output; export a detection model without embedded NMS")\n    tensors = [graph.Tensors(graph.Inputs(0)), graph.Tensors(graph.Outputs(0))]\n    shapes = [[t.Shape(i) for i in range(t.ShapeLength())] for t in tensors]\n    if shapes[0] != [1, 320, 320, 3] or shapes[1] not in ([1, 6, 2100], [1, 2100, 6]):\n        raise ValueError(f"Unexpected custom model shapes: {shapes}")\n    descriptions = []\n    for tensor, shape in zip(tensors, shapes):\n        if tensor.Type() not in (schema.TensorType.INT8, schema.TensorType.UINT8):\n            raise ValueError("Use full_integer_quant.tflite, with integer input AND output")\n        q = tensor.Quantization()\n        if q is None or q.ScaleLength() != 1 or q.ZeroPointLength() != 1 or not q.Scale(0) > 0:\n            raise ValueError("Expected per-tensor input/output quantization")\n        descriptions.append({"shape": shape, "type": "int8" if tensor.Type() == schema.TensorType.INT8 else "uint8",\n                             "scale": float(q.Scale(0)), "zero_point": int(q.ZeroPoint(0))})\n    has_tpu = any(model.OperatorCodes(i).CustomCode() == b"edgetpu-custom-op" for i in range(model.OperatorCodesLength()))\n    if has_tpu != compiled:\n        raise ValueError("Unexpected presence/absence of the compiled Edge TPU operation")\n    return {"input": descriptions[0], "output": descriptions[1], "edge_tpu_compiled": has_tpu}\n\n\ndef prepare_inputs(training_zip, dataset_zip, output):\n    """Tie the supplied dataset to the model\'s own manifest; never calibrate on VAL."""\n    import yaml\n    from dataset_tools import validate_dataset\n    extract(training_zip, output / "training_result")\n    extract(dataset_zip, output / "dataset")\n    training = output / "training_result"\n    contract = json.loads((training / "model_contract.json").read_text())\n    if {int(k): v for k, v in contract["classes"].items()} != NAMES or contract["training_imgsz"] != 320:\n        raise ValueError("Checkpoint contract must be figure=0, stop_sign=1, trained at 320")\n    weights = training / "weights/best.pt"\n    if sha(weights) != contract["best_pt_sha256"]:\n        raise ValueError("Checkpoint SHA-256 does not match its training contract")\n    roots = [p.parent for p in (output / "dataset").rglob("data.yaml") if (p.parent / "manifest.json").is_file()]\n    if len(roots) != 1:\n        raise ValueError("Expected one prepared dataset with manifest.json")\n    dataset = roots[0]\n    summary = validate_dataset(dataset)\n    manifest = json.loads((dataset / "manifest.json").read_text())\n    trained_manifest = json.loads((training / "dataset_manifest.json").read_text())\n    if manifest != trained_manifest:\n        raise ValueError("The supplied dataset is not the dataset recorded by this training run")\n    for record in manifest["images"]:\n        path = (dataset / record["image"]).resolve()\n        if not path.is_relative_to(dataset.resolve()) or sha(path) != record["sha256"]:\n            raise ValueError("Dataset image differs from the training manifest")\n    # Ultralytics 8.3.228 defaults to val when collecting INT8 examples. This\n    # separate YAML intentionally maps BOTH entries to TRAIN, so VAL never leaks.\n    calibration = {"path": str(dataset), "train": "images/train", "val": "images/train", "names": NAMES}\n    (output / "calibration_train_only.yaml").write_text(yaml.safe_dump(calibration, sort_keys=False))\n    validation = {**calibration, "val": "images/val"}\n    (output / "validation.yaml").write_text(yaml.safe_dump(validation, sort_keys=False))\n    shutil.copyfile(weights, output / (STEM + ".pt"))\n    print(f"Verified model {sha(weights)}; calibrating on {summary[\'train\'][\'images\']} TRAIN images only.", flush=True)\n    return {"weights_sha256": sha(weights), "dataset_zip_sha256": sha(dataset_zip),\n            "training_zip_sha256": sha(training_zip), "calibration_images": summary["train"]["images"],\n            "validation_images": summary["val"]["images"], "evaluation_note": manifest.get("evaluation_note", "")}\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--training-zip", type=Path, required=True)\n    parser.add_argument("--dataset-zip", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--compile-only", action="store_true", help="Resume a verified completed INT8 conversion after compiler failure")\n    args = parser.parse_args()\n    if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):\n        parser.error("Use x86-64 Linux/Colab for conversion; run the compiled result on the Pi.")\n    os.environ["YOLO_AUTOINSTALL"] = "false"\n    from ultralytics import YOLO, __version__\n    if __version__ != "8.3.228":\n        raise ValueError("Use the included pinned export requirements")\n    output = args.output.resolve()\n    output.mkdir(parents=True, exist_ok=True)\n    record = output / "conversion_record.json"\n    source = output / (STEM + "_saved_model") / (STEM + "_full_integer_quant.tflite")\n    if args.compile_only:\n        provenance = json.loads(record.read_text())\n        if sha(args.training_zip) != provenance["training_zip_sha256"] or sha(args.dataset_zip) != provenance["dataset_zip_sha256"] or sha(source) != provenance["int8_sha256"]:\n            raise ValueError("Resume inputs differ from the completed INT8 conversion")\n    else:\n        provenance = prepare_inputs(args.training_zip, args.dataset_zip, output)\n        model = YOLO(str(output / (STEM + ".pt")))\n        if model.task != "detect" or model.names != NAMES:\n            raise ValueError(f"Unexpected trained classes/task: {model.names}, {model.task}")\n        # TFLite export accepts int8=True; the edgetpu export entry point does not\n        # accept it in 8.3.228. Select the FULL integer artifact explicitly below.\n        model.export(format="tflite", int8=True, imgsz=320, batch=1, nms=False,\n                     data=str(output / "calibration_train_only.yaml"), device="cpu")\n        inspect_model(source)\n        provenance["int8_sha256"] = sha(source)\n        record.write_text(json.dumps(provenance, indent=2))\n    compiler, compiler_version = ensure_compiler(output)\n    result = subprocess.run([str(compiler), "--out_dir", str(source.parent), "--show_operations",\n                             "--search_delegate", "--delegate_search_step", "30", "--timeout_sec", "180",\n                             "--min_runtime_version", "14", str(source)], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)\n    print(result.stdout, flush=True)\n    (output / "compiler_stdout.txt").write_text(result.stdout)\n    result.check_returncode()\n    compiled = source.with_name(source.stem + "_edgetpu.tflite")\n    tensors = inspect_model(compiled, compiled=True)\n    # Evaluate uncompiled full-INT8 on CPU and PyTorch at identical image/batch\n    # settings. The compiled model still needs a real --check-coral/live Pi run.\n    report = {}\n    for label, path in (("pytorch", output / (STEM + ".pt")), ("int8_cpu", source)):\n        metrics = YOLO(str(path), task="detect").val(data=str(output / "validation.yaml"), split="val",\n                    imgsz=320, rect=False, batch=1, device="cpu", workers=0, plots=True,\n                    project=str(output / "validation"), name=label)\n        report[label] = {k: float(v) for k, v in metrics.results_dict.items()}\n    (output / "quantization_validation.json").write_text(json.dumps(report, indent=2))\n    model_dir = output / "pi_models"\n    model_dir.mkdir(exist_ok=True)\n    destination = model_dir / (STEM + "_edgetpu.tflite")\n    shutil.copyfile(compiled, destination)\n    manifest = {**provenance, **tensors, "model": destination.name, "sha256": sha(destination),\n                "class_count": 2, "classes": NAMES, "target_class": 0, "stop_sign_class": 1,\n                "box_coordinates": "normalized", "nms_in_graph": False, "ultralytics": __version__,\n                "compiler_version": compiler_version, "compiler_archive_sha256": COMPILER_SHA256,\n                "min_runtime_version": 14, "pi_hardware_tested": False, "validation": report}\n    (model_dir / (STEM + "_manifest.json")).write_text(json.dumps(manifest, indent=2))\n    (model_dir / (STEM + "_edgetpu.sha256")).write_text(f"{sha(destination)}  {destination.name}\\n")\n    (model_dir / "figure_labels.txt").write_text("0 figure\\n1 stop_sign\\n")\n    bundle = output / "figure_coral_model.zip"\n    with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(model_dir.iterdir()):\n            archive.write(path, "models/" + path.name)\n        archive.write(output / "compiler_stdout.txt", "evidence/compiler_stdout.txt")\n        archive.write(output / "quantization_validation.json", "evidence/quantization_validation.json")\n        for path in source.parent.glob("*edgetpu.log"):\n            archive.write(path, "evidence/" + path.name)\n        for path in (output / "validation").rglob("*.png"):\n            archive.write(path, "evidence/validation/" + path.relative_to(output / "validation").as_posix())\n    with zipfile.ZipFile(bundle) as archive:\n        assert archive.testzip() is None\n    print("READY:", bundle, flush=True)\n    print("Next: compare PyTorch/INT8 metrics, transfer models/, then run --yolo-profile figure --check-coral on the Pi.")\n\n\nif __name__ == "__main__":\n    main()\n', 'export_yolo.py': '#!/usr/bin/env python3\n"""Export the pinned stock YOLOv8n COCO detector on x86_64 Linux, not on the Pi.\n\nPython is only a model-preparation dependency. The live application, inference,\nletterboxing, decoding, NMS, tracking, planning, display, and logs are C++.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport io\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport tarfile\nimport urllib.request\n\nWEIGHTS_URL = "https://github.com/ultralytics/assets/releases/download/v8.3.0/yolov8n.pt"\nWEIGHTS_SHA256 = "f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36"\nCOMPILER_URL = "https://github.com/ultralytics/assets/releases/download/v0.0.0/edgetpu-compiler_16.0_amd64.tar.gz"\nCOMPILER_SHA256 = "f9a97adbf2ff621d24a91721191264f2abc3c1db31e63d9a266d988f907269ce"\n\n\ndef ensure_compiler(output):\n    """Install a verified private compiler; the Coral apt repository is unavailable.\n\n    Ultralytics distributes the original compiler 16 package files. Keep its\n    bundled loader/libraries together, verify the archive before extraction,\n    and prepend a quoted launcher to PATH for the pinned Ultralytics exporter.\n    No sudo, apt source changes, GPG prompts, or system installation are needed.\n    """\n    root = (output / ".compiler16").resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    archive_path = root / "compiler.tar.gz"\n    if not archive_path.is_file():\n        temporary = archive_path.with_suffix(".gz.part")\n        urllib.request.urlretrieve(COMPILER_URL, temporary)\n        if hashlib.sha256(temporary.read_bytes()).hexdigest() != COMPILER_SHA256:\n            raise RuntimeError("Downloaded compiler failed SHA-256 validation")\n        temporary.replace(archive_path)\n    data = archive_path.read_bytes()\n    if hashlib.sha256(data).hexdigest() != COMPILER_SHA256:\n        raise RuntimeError("Cached compiler archive failed SHA-256 validation")\n    with tarfile.open(fileobj=io.BytesIO(data), mode="r:gz") as archive:\n        for member in archive.getmembers():\n            relative = Path(member.name)\n            if relative.is_absolute() or ".." in relative.parts:\n                raise RuntimeError("Unsafe compiler archive path")\n            target = root / relative\n            if not target.resolve().is_relative_to(root):\n                raise RuntimeError("Compiler extraction would leave its directory")\n            if member.isdir():\n                target.mkdir(parents=True, exist_ok=True)\n            elif member.isfile():\n                target.parent.mkdir(parents=True, exist_ok=True)\n                target.write_bytes(archive.extractfile(member).read())\n                target.chmod(0o755 if "bin" in relative.parts else 0o644)\n            else:\n                raise RuntimeError("Unexpected compiler archive member")\n    launcher = root / "edgetpu_compiler"\n    launcher.write_text(\n        \'#!/bin/sh\\n\'\n        \'bundle=$(CDPATH= cd -- "$(dirname -- "$0")/usr/bin/edgetpu_compiler_bin" && pwd)\\n\'\n        \'exec "$bundle/ld-linux-x86-64.so.2" --library-path "$bundle" "$bundle/edgetpu_compiler" "$@"\\n\',\n        encoding="utf-8",\n    )\n    launcher.chmod(0o755)\n    version = subprocess.check_output([str(launcher), "--version"], text=True).strip()\n    if "16.0.384591198" not in version:\n        raise RuntimeError(f"Unexpected compiler version: {version}")\n    os.environ["PATH"] = str(root) + os.pathsep + os.environ.get("PATH", "")\n    print(version, flush=True)\n    return launcher, version\n\ndef inspect_compiled_model(path):\n    """Read the FlatBuffer contract without trying to execute an Edge TPU on this computer."""\n    from tensorflow.lite.python import schema_py_generated as schema\n    data = path.read_bytes()\n    if data[4:8] != b"TFL3":\n        raise RuntimeError("Exporter did not produce a TFLite FlatBuffer")\n    model = schema.Model.GetRootAsModel(data, 0)\n    graph = model.Subgraphs(0)\n    if graph.InputsLength() != 1 or graph.OutputsLength() != 1:\n        raise RuntimeError("Expected one input and one output; use COCO detect with nms=False")\n    input_tensor = graph.Tensors(graph.Inputs(0))\n    output_tensor = graph.Tensors(graph.Outputs(0))\n    input_shape = [input_tensor.Shape(i) for i in range(input_tensor.ShapeLength())]\n    output_shape = [output_tensor.Shape(i) for i in range(output_tensor.ShapeLength())]\n    if input_shape != [1, 320, 320, 3] or output_shape not in ([1, 84, 2100], [1, 2100, 84]):\n        raise RuntimeError(f"Unexpected model shapes: {input_shape}, {output_shape}")\n    quantized = (schema.TensorType.INT8, schema.TensorType.UINT8)\n    if input_tensor.Type() not in quantized:\n        raise RuntimeError("Expected quantized int8/uint8 input")\n    if not any(model.OperatorCodes(i).CustomCode() == b"edgetpu-custom-op" for i in range(model.OperatorCodesLength())):\n        raise RuntimeError("Compilation did not produce an Edge TPU operation")\n    return {"input_shape": input_shape, "output_shape": output_shape,\n            "input_tensor_type": int(input_tensor.Type()), "output_tensor_type": int(output_tensor.Type())}\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output", type=Path, default=Path("models"))\n    parser.add_argument("--compile-only", action="store_true",\n                        help="Reuse this export\'s existing full_integer_quant.tflite after a compiler failure")\n    args = parser.parse_args()\n    if platform.system() != "Linux" or platform.machine() not in ("x86_64", "AMD64"):\n        parser.error("Export requires x86_64 Linux or Google Colab. Run the compiled model on the Pi afterward.")\n    from ultralytics import YOLO, __version__\n    if __version__ != "8.3.228":\n        raise RuntimeError("Use ultralytics==8.3.228 for this evaluation export contract")\n    args.output.mkdir(parents=True, exist_ok=True)\n    weights = args.output / "yolov8n.pt"\n    if not weights.exists():\n        temporary = weights.with_suffix(".pt.part")\n        urllib.request.urlretrieve(WEIGHTS_URL, temporary)\n        if hashlib.sha256(temporary.read_bytes()).hexdigest() != WEIGHTS_SHA256:\n            raise RuntimeError("Downloaded YOLO weights failed SHA-256 validation")\n        temporary.replace(weights)\n    if hashlib.sha256(weights.read_bytes()).hexdigest() != WEIGHTS_SHA256:\n        raise RuntimeError("Existing weights do not match the pinned stock COCO model")\n    model = YOLO(str(weights))\n    if len(model.names) != 80 or model.names[0] != "person" or model.names[11] != "stop sign":\n        raise RuntimeError("Expected COCO-80 class mapping")\n    compiler, compiler_version = ensure_compiler(args.output)\n    if args.compile_only:\n        # Reuse only the known output of this recipe, not an arbitrary model.\n        source = args.output / "yolov8n_saved_model/yolov8n_full_integer_quant.tflite"\n        if not source.is_file():\n            raise RuntimeError("No completed INT8 conversion found. Run without --compile-only first.")\n        subprocess.run([str(compiler), "--out_dir", str(source.parent), "--show_operations",\n                        "--search_delegate", "--delegate_search_step", "30", "--timeout_sec", "180",\n                        "--min_runtime_version", "14", str(source)], check=True)\n        exported = source.with_name(source.stem + "_edgetpu.tflite")\n    else:\n        # Edge TPU export enables INT8 internally; 8.3.228 rejects an explicit int8=True.\n        exported = Path(model.export(format="edgetpu", imgsz=320, batch=1,\n                                     nms=False, data="coco128.yaml", device="cpu"))\n    contract = inspect_compiled_model(exported)\n    destination = args.output / "yolov8n_320_edgetpu.tflite"\n    shutil.copyfile(exported, destination)\n    digest = hashlib.sha256(destination.read_bytes()).hexdigest()\n    manifest = {"model": destination.name, "sha256": digest, "weights_sha256": WEIGHTS_SHA256,\n                "ultralytics": __version__, "compiler_version": compiler_version,\n                "compiler_archive_sha256": COMPILER_SHA256, "min_runtime_version": 14,\n                "class_count": 80, "person_class": 0, "stop_sign_class": 11,\n                "box_coordinates": "normalized", "nms_in_graph": False, **contract}\n    (args.output / "yolov8n_320_manifest.json").write_text(json.dumps(manifest, indent=2) + "\\n")\n    (args.output / "yolov8n_320_edgetpu.sha256").write_text(f"{digest}  yolov8n_320_edgetpu.tflite\\n")\n    for log in exported.parent.glob("*edgetpu.log"):\n        shutil.copyfile(log, args.output / "yolov8n_320_compile.log")\n    print(f"Ready: {destination}. Inspect the compiler log for CPU operations; this is not a Pi benchmark.")\n\nif __name__ == "__main__":\n    main()\n', 'dataset_tools.py': '"""Prepare camera recordings and reviewed CVAT labels for this two-class experiment.\n\nRun --help for commands. Python is used offline for dataset preparation; this\ndoes not change the Raspberry Pi\'s native C++ application.\n"""\nfrom pathlib import Path\nimport argparse\nimport collections\nimport hashlib\nimport json\nimport math\nimport re\nimport shutil\nimport stat\nimport sys\nimport zipfile\n\nNAMES = {0: "figure", 1: "stop_sign"}\nIMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}\n\n\ndef recording(name):\n    """Keep every frame from one recording in its preselected split."""\n    match = re.match(r"^((train|val|test)\\d+)_", name)\n    if not match:\n        raise ValueError(f"Image name must start train01_, val01_, test01_, etc.: {name}")\n    return match.group(1), match.group(2)\n\n\ndef names_from_yaml(path):\n    import yaml\n    data = yaml.safe_load(path.read_text(encoding="utf-8"))\n    names = data.get("names", {})\n    if isinstance(names, list):\n        names = dict(enumerate(names))\n    names = {int(key): value for key, value in names.items()}\n    if len(names) != 2 or set(names.values()) != set(NAMES.values()):\n        raise ValueError(f"Expected only figure and stop_sign classes in {path}; got {names}")\n    return names\n\n\ndef read_labels(path, mapping=None):\n    """Reject malformed, out-of-frame, or non-detection annotations."""\n    rows = []\n    for line_number, line in enumerate(path.read_text(encoding="utf-8-sig").splitlines(), 1):\n        if not line.strip():\n            continue\n        values = line.split()\n        if len(values) != 5:\n            raise ValueError(f"{path}:{line_number}: need class cx cy width height (5 columns)")\n        category = int(values[0])\n        if mapping is not None:\n            if category not in mapping:\n                raise ValueError(f"Unknown class {category} in {path}")\n            category = mapping[category]\n        if category not in NAMES:\n            raise ValueError(f"Unsupported class {category} in {path}")\n        x, y, width, height = map(float, values[1:])\n        if not all(math.isfinite(v) for v in (x, y, width, height)):\n            raise ValueError(f"Nonfinite label in {path}")\n        if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < width <= 1 and 0 < height <= 1):\n            raise ValueError(f"Coordinates must be normalized to [0,1] in {path}")\n        if x-width/2 < -1e-5 or y-height/2 < -1e-5 or x+width/2 > 1+1e-5 or y+height/2 > 1+1e-5:\n            raise ValueError(f"Box extends outside the image in {path}")\n        rows.append((category, x, y, width, height))\n    return rows\n\n\ndef validate_dataset(root):\n    """Validate images and labels; an explicit manifest permits a within-recording holdout."""\n    from PIL import Image\n    import yaml\n    root = Path(root)\n    if names_from_yaml(root/"data.yaml") != NAMES:\n        raise ValueError("Prepared class order must be 0=figure, 1=stop_sign")\n    config = yaml.safe_load((root/"data.yaml").read_text(encoding="utf-8"))\n    manifest_path = root/"manifest.json"\n    manifest = json.loads(manifest_path.read_text(encoding="utf-8")) if manifest_path.is_file() else {}\n    policy = manifest.get("split_policy", "recording_prefix")\n    if policy not in ("recording_prefix", "explicit_holdout"):\n        raise ValueError(f"Unknown split policy: {policy}")\n    entries = manifest.get("images", [])\n    expected = {item["image"]: item for item in entries}\n    if policy == "explicit_holdout" and (not expected or len(expected) != len(entries)):\n        raise ValueError("An explicit holdout requires a complete, unique image manifest")\n    seen_pixels, summary, seen_paths, seen_stems = {}, {}, set(), set()\n    for split in ("train", "val", "test"):\n        images = sorted(p for p in (root/"images"/split).glob("*") if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)\n        if split == "test" and not images and not config.get("test"):\n            continue  # A test set is optional; never substitute validation for it.\n        if not images:\n            raise ValueError(f"No images in required {split} split")\n        counts = collections.Counter()\n        groups, negatives = set(), 0\n        for image in images:\n            group, declared_split = recording(image.name)\n            relative = image.relative_to(root).as_posix()\n            seen_paths.add(relative)\n            if image.stem in seen_stems:\n                raise ValueError(f"An image filename is reused across splits: {image.name}")\n            seen_stems.add(image.stem)\n            if policy == "explicit_holdout":\n                item = expected.get(relative)\n                if not item or item.get("split") != split or item.get("recording") != group:\n                    raise ValueError(f"Image does not match the explicit split manifest: {image}")\n                if item.get("sha256") != hashlib.sha256(image.read_bytes()).hexdigest():\n                    raise ValueError(f"Image content differs from split manifest: {image}")\n            elif declared_split != split:\n                raise ValueError(f"Recording moved across splits: {image}")\n            groups.add(group)\n            with Image.open(image) as opened:\n                if opened.size != (640, 480):\n                    raise ValueError(f"Expected the 640x480 camera image, not a screen recording: {image}")\n                digest = hashlib.sha256(opened.convert("RGB").tobytes()).hexdigest()\n            if digest in seen_pixels and seen_pixels[digest][0] != split:\n                raise ValueError(f"Identical pixels cross splits: {image} and {seen_pixels[digest][1]}")\n            seen_pixels[digest] = (split, str(image))\n            label = root/"labels"/split/(image.stem+".txt")\n            if not label.is_file():\n                raise ValueError(f"Missing label {label}; reviewed background images need an empty label file")\n            rows = read_labels(label)\n            negatives += not rows\n            counts.update(row[0] for row in rows)\n        if any(counts[category] == 0 for category in NAMES):\n            raise ValueError(f"Both classes need examples in {split}: {counts}")\n        summary[split] = {"images": len(images), "backgrounds": negatives,\n                          "boxes": {NAMES[k]: counts[k] for k in NAMES}, "recordings": sorted(groups)}\n    if policy == "explicit_holdout" and seen_paths != set(expected):\n        raise ValueError("Split manifest contains missing or extra image entries")\n    return summary\n\n\ndef safe_extract(archive_path, destination):\n    """Extract a supplied dataset into a new folder without path traversal or overwrites."""\n    destination = Path(destination).resolve()\n    destination.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(archive_path) as archive:\n        if sum(item.file_size for item in archive.infolist()) > 8_000_000_000:\n            raise ValueError("Dataset expands beyond this notebook\'s 8 GB limit")\n        for item in archive.infolist():\n            name = item.filename.replace("\\\\", "/")\n            relative = Path(name)\n            if relative.is_absolute() or ".." in relative.parts or ":" in name or stat.S_ISLNK(item.external_attr >> 16):\n                raise ValueError(f"Unsafe ZIP entry: {name}")\n            target = destination/relative\n            if not target.resolve().is_relative_to(destination):\n                raise ValueError(f"ZIP entry escapes destination: {name}")\n            if item.is_dir():\n                target.mkdir(parents=True, exist_ok=True)\n            else:\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with archive.open(item) as source, target.open("xb") as output:\n                    shutil.copyfileobj(source, output)\n\n\ndef extract(args):\n    """Decode at capture rate and keep one frame per interval; skip exposure startup."""\n    import cv2\n    if args.fps <= 0 or args.every <= 0:\n        raise ValueError("fps and every must be positive")\n    videos = sorted(args.videos.glob("*.mjpeg"))\n    if not videos:\n        raise ValueError("No .mjpeg recordings found")\n    args.out.mkdir(parents=True, exist_ok=True)\n    for video in videos:\n        if not re.fullmatch(r"(train|val|test)\\d+", video.stem):\n            raise ValueError(f"Name recordings train01.mjpeg, val01.mjpeg, test01.mjpeg, etc.: {video}")\n        if any(args.out.glob(video.stem+"_*")):\n            raise ValueError(f"Frames for {video.stem} already exist; use a new output folder or recording name")\n        capture = cv2.VideoCapture(str(video))\n        if not capture.isOpened():\n            raise ValueError(f"Cannot open {video}")\n        frame_number, saved = 0, 0\n        stride = max(1, round(args.fps*args.every))\n        skip = round(args.fps*2)  # Let camera exposure settle for two seconds.\n        while True:\n            ok, frame = capture.read()\n            if not ok:\n                break\n            if frame.shape != (480, 640, 3):\n                raise ValueError(f"Expected 640x480 camera video: {video}")\n            if frame_number >= skip and (frame_number-skip) % stride == 0:\n                target = args.out/f"{video.stem}_f{frame_number:06d}.jpg"\n                if not cv2.imwrite(str(target), frame, [cv2.IMWRITE_JPEG_QUALITY, 95]):\n                    raise IOError(f"Cannot write {target}")\n                saved += 1\n            frame_number += 1\n        capture.release()\n        if not saved:\n            raise ValueError(f"No usable frames from {video}")\n        print(f"{video.name}: {saved} images; keep/review a varied subset before labeling")\n\n\ndef prepare(args):\n    """Normalize CVAT labels using recording prefixes or an explicit reviewed split plan."""\n    import yaml\n    candidates = [p for p in args.source.rglob("data.yaml") if (p.parent/"images").is_dir()]\n    if len(candidates) != 1:\n        raise ValueError("Expected one exported data.yaml alongside images/ and labels/")\n    origin = candidates[0].parent\n    source_names = names_from_yaml(candidates[0])\n    mapping = {source_id: next(k for k,v in NAMES.items() if v==name) for source_id,name in source_names.items()}\n    reviewed = set()\n    if args.reviewed_backgrounds:\n        reviewed = {line.strip() for line in args.reviewed_backgrounds.read_text().splitlines() if line.strip()}\n    images = sorted(p for p in (origin/"images").rglob("*") if p.suffix.lower() in IMAGE_SUFFIXES)\n    planned, seen_names = [], set()\n    for image in images:\n        group, split = recording(image.name)\n        if image.stem in seen_names:\n            raise ValueError(f"Duplicate image stem: {image.stem}")\n        seen_names.add(image.stem)\n        label = origin/"labels"/image.relative_to(origin/"images").with_suffix(".txt")\n        if label.is_file():\n            rows = read_labels(label, mapping)\n        elif image.name in reviewed:\n            rows = []  # The user explicitly reviewed this frame as containing neither class.\n        else:\n            raise ValueError(f"Missing label for {image.name}. Label it, or list this exact filename in reviewed_backgrounds.txt if BOTH target classes are absent.")\n        planned.append((image, group, split, rows))\n    if not planned:\n        raise ValueError("No exported images; export the CVAT dataset WITH images")\n    split_policy, split_note = "recording_prefix", "Recordings assigned by their original filename prefixes."\n    split_plan_path = getattr(args, "split_plan", None)\n    if split_plan_path:\n        plan = json.loads(split_plan_path.read_text(encoding="utf-8"))\n        assignments = plan.get("assignments", {})\n        if set(assignments) != {image.name for image, _, _, _ in planned}:\n            raise ValueError("Split plan must assign every source image exactly once by its original filename")\n        if any(split not in ("train", "val", "test") for split in assignments.values()):\n            raise ValueError("Split assignments must be train, val, or test")\n        planned = [(image, group, assignments[image.name], rows) for image, group, _, rows in planned]\n        split_policy = "explicit_holdout"\n        split_note = plan.get("evaluation_note", "Exploratory split; source recordings may span training and validation.")\n        intended_counts = plan.get("target_counts")\n        actual_counts = dict(collections.Counter(split for _, _, split, _ in planned))\n        if intended_counts is not None and intended_counts != actual_counts:\n            raise ValueError(f"Split plan counts differ: expected {intended_counts}, found {actual_counts}")\n    # Check coverage before creating output, so missing recordings leave no partial dataset.\n    split_sizes = collections.Counter(split for _, _, split, _ in planned)\n    required_splits = ("train", "val") + (("test",) if split_sizes["test"] else ())\n    missing_splits = [split for split in required_splits if not split_sizes[split]]\n    if missing_splits:\n        counts = ", ".join(f"{split}={split_sizes[split]}" for split in required_splits)\n        recordings = ", ".join(f"{split}01.mjpeg" for split in missing_splits)\n        raise ValueError(\n            f"Dataset is incomplete ({counts}). Missing splits: {\', \'.join(missing_splits)}. "\n            f"Add labeled sessions named {recordings}, or supply a reviewed --split-plan "\n            "to reserve existing images for validation. Test images are optional. No output was written."\n        )\n    for split in required_splits:\n        classes = {row[0] for _, _, selected_split, rows in planned if selected_split == split for row in rows}\n        missing_classes = [NAMES[category] for category in NAMES if category not in classes]\n        if missing_classes:\n            raise ValueError(\n                f"The {split} split has no labeled examples of: {\', \'.join(missing_classes)}. "\n                "Record and label these objects in this split before preparation. No output was written."\n            )\n    if args.out.exists() or args.out.with_suffix(".zip").exists():\n        raise ValueError("Output dataset/ZIP already exists; choose a new --out name")\n    manifest = []\n    for image, group, split, rows in planned:\n        dest = args.out/"images"/split/image.name\n        label = args.out/"labels"/split/(image.stem+".txt")\n        dest.parent.mkdir(parents=True, exist_ok=True)\n        label.parent.mkdir(parents=True, exist_ok=True)\n        shutil.copyfile(image, dest)\n        label.write_text("".join(str(row[0])+" "+" ".join(f"{v:.8f}" for v in row[1:])+"\\n" for row in rows))\n        manifest.append({"image": dest.relative_to(args.out).as_posix(), "recording": group,\n                         "split": split, "sha256": hashlib.sha256(dest.read_bytes()).hexdigest()})\n    data = {"path": ".", "train": "images/train", "val": "images/val", "names": NAMES}\n    if split_sizes["test"]:\n        data["test"] = "images/test"\n    (args.out/"data.yaml").write_text(yaml.safe_dump(data, sort_keys=False))\n    metadata = {"images": manifest, "split_policy": split_policy, "evaluation_note": split_note}\n    if split_plan_path:\n        metadata["split_plan"] = plan\n    # The validator checks this assignment manifest without changing original image names.\n    (args.out/"manifest.json").write_text(json.dumps(metadata, indent=2)+"\\n", encoding="utf-8")\n    summary = validate_dataset(args.out)\n    metadata["summary"] = summary\n    (args.out/"manifest.json").write_text(json.dumps(metadata, indent=2)+"\\n", encoding="utf-8")\n    with zipfile.ZipFile(args.out.with_suffix(".zip"), "w", zipfile.ZIP_DEFLATED) as archive:\n        for file in sorted(args.out.rglob("*")):\n            if file.is_file():\n                archive.write(file, Path(args.out.name)/file.relative_to(args.out))\n    print(json.dumps(summary, indent=2))\n    print(f"Ready: {args.out.with_suffix(\'.zip\')}")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest="command", required=True)\n    extract_parser = sub.add_parser("extract", help="Sample camera MJPEG recordings for annotation")\n    extract_parser.add_argument("--videos", type=Path, required=True)\n    extract_parser.add_argument("--out", type=Path, default=Path("images_to_label"))\n    extract_parser.add_argument("--fps", type=float, default=15, help="Capture frame rate; raw MJPEG has no reliable timing metadata")\n    extract_parser.add_argument("--every", type=float, default=1, help="Seconds between sampled frames")\n    prepare_parser = sub.add_parser("prepare", help="Normalize a reviewed CVAT dataset and create its ZIP")\n    prepare_parser.add_argument("--source", type=Path, required=True)\n    prepare_parser.add_argument("--out", type=Path, default=Path("tabletop_dataset"))\n    prepare_parser.add_argument("--reviewed-backgrounds", type=Path)\n    prepare_parser.add_argument("--split-plan", type=Path, help="JSON assignments for an explicitly chosen train/val holdout; filenames stay unchanged")\n    validate_parser = sub.add_parser("validate", help="Check an already prepared dataset")\n    validate_parser.add_argument("root", type=Path)\n    args = parser.parse_args()\n    if args.command == "extract":\n        extract(args)\n    elif args.command == "prepare":\n        prepare(args)\n    else:\n        print(json.dumps(validate_dataset(args.root), indent=2))\n\n\nif __name__ == "__main__":\n    try:\n        main()\n    except (ValueError, OSError) as error:\n        print(f"ERROR: {error}", file=sys.stderr)\n        raise SystemExit(1)\n', 'requirements-export.txt': '# Model preparation on x86_64 Linux / Python 3.12; not the live Pi application.\nultralytics==8.3.228\ntensorflow==2.19.0\ntf-keras==2.19.0\nnumpy==2.1.3\nopencv-python==4.11.0.86\n# Explicit conversion dependencies required by the pinned Ultralytics exporter.\n# onnx2tf 1.28.2 does not declare its supporting packages in wheel metadata.\nonnx2tf==1.28.2\nonnx==1.17.0\nonnx-graphsurgeon==0.5.8\nsng4onnx==1.0.4\nai-edge-litert==1.2.0\nonnxslim==0.1.71\nonnxruntime==1.22.1\nprotobuf==5.29.5\n'}

for name, source in SOURCES.items():
    (TOOLS / name).write_text(source, encoding="utf8")
VENV = Path("/content/figure_export_env")
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(VENV)], check=True)
PYTHON = VENV / "bin/python"
EXPORT_ENV = dict(os.environ, YOLO_AUTOINSTALL="false", TF_CPP_MIN_LOG_LEVEL="2")
def run_logged(command, log):
    with Path(log).open("w") as stream:
        with subprocess.Popen([str(item) for item in command], stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1, env=EXPORT_ENV) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                stream.write(line)
            status = process.wait()
    if status:
        raise RuntimeError(f"Command failed with status {status}; inspect the first error above. Full log: {log}")
run_logged([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--disable-pip-version-check", "-r", TOOLS / "requirements-export.txt"], TOOLS / "install.log")
probe = "import numpy as np; import numpy._core.strings; import torch, tensorflow, tf_keras, onnx, onnx2tf, onnx_graphsurgeon, sng4onnx, onnxslim, onnxruntime, ai_edge_litert, ultralytics; print('NumPy', np.__version__, 'TF', tensorflow.__version__, 'YOLO', ultralytics.__version__); print(np.strings.center(['ok'], 6)); assert ultralytics.__version__ == '8.3.228'"
run_logged([PYTHON, "-c", probe], TOOLS / "import_check.log")
print("Export environment ready. Continue below; no notebook restart is needed.")

## Select the trained model and matching dataset
Use the two ZIP files from this training run. Their checkpoint hash, classes,
image hashes, and dataset manifest are checked before conversion.

In [ ]:
from google.colab import files
TRAINING_ZIP = Path("/content/figure_training_result.zip")
DATASET_ZIP = Path("/content/tabletop_dataset.zip")
if not TRAINING_ZIP.is_file() or not DATASET_ZIP.is_file():
    print("Select figure_training_result.zip AND tabletop_dataset.zip.")
    uploaded = files.upload()
    import zipfile
    for filename in uploaded:
        path = Path(filename).resolve()
        if not zipfile.is_zipfile(path):
            continue
        with zipfile.ZipFile(path) as archive:
            names = set(archive.namelist())
        if "weights/best.pt" in names and "model_contract.json" in names:
            TRAINING_ZIP = path
        elif "tabletop_dataset/manifest.json" in names:
            DATASET_ZIP = path
assert TRAINING_ZIP.is_file() and DATASET_ZIP.is_file(), "Both ZIPs are required."
print("Training results:", TRAINING_ZIP)
print("Dataset:", DATASET_ZIP)

## Convert, compile, and validate
This takes several minutes. The exporter deliberately selects the
`full_integer_quant.tflite` artifact, since the other INT8 artifact has float
input/output. Quantization calibration uses a dedicated YAML that points only
to training images; the original validation split stays separate.

If compilation fails **after** `conversion_record.json` was saved, set
`RESUME_COMPILE=True` below and rerun this cell to reuse that verified INT8
conversion. If conversion itself failed, fix the error and choose a fresh
`OUTPUT` folder. Do not simply rename `best.pt` or another TFLite file.

In [ ]:
OUTPUT = Path("/content/figure_coral_export")
RESUME_COMPILE = False
command = [PYTHON, "-u", TOOLS / "export_figure.py", "--training-zip", TRAINING_ZIP,
           "--dataset-zip", DATASET_ZIP, "--output", OUTPUT]
if RESUME_COMPILE:
    command.append("--compile-only")
run_logged(command, TOOLS / "export.log")

## Review and download
Compare the PyTorch and INT8 metrics before using the model. A large recall/mAP
drop means the conversion needs investigation. These measurements use CPU
inference of the uncompiled model; the actual compiled file still needs the
Pi's `--check-coral` and live evaluation. The compiler operation counts show
where work runs, not a promised frame rate.

In [ ]:
import shutil, zipfile
report = json.loads((OUTPUT / "quantization_validation.json").read_text())
print(json.dumps(report, indent=2))
manifest = json.loads((OUTPUT / "pi_models/figure_yolov8n_320_manifest.json").read_text())
print("Input/output:", manifest["input"], manifest["output"])
print("Model SHA-256:", manifest["sha256"])
DOWNLOAD = Path("/content/figure_coral_model.zip")
shutil.copyfile(OUTPUT / DOWNLOAD.name, DOWNLOAD)
with zipfile.ZipFile(DOWNLOAD) as archive:
    assert archive.testzip() is None
print("READY:", DOWNLOAD, DOWNLOAD.stat().st_size, "bytes")
print("If the download does not start, use Files > right-click figure_coral_model.zip > Download.")
files.download(str(DOWNLOAD))

## Use the updated C++ project
Extract this model ZIP in the **updated figure evaluation project**, which
places the `.tflite` and SHA-256 file under `models/`. Then run:
```bash
(cd models && sha256sum -c figure_yolov8n_320_edgetpu.sha256)
./build/tabletop_delivery --yolo-profile figure --check-coral
./build/tabletop_delivery --yolo-profile figure --tracking compare
```
Keep the blue marker for the first comparison. When the figure box and LiDAR
position are reliable, use `--tracking yolo` for model-controlled tracking.
Class 0 is the figure; class 1 is the stop sign. Stop-sign confirmation remains
at confidence 0.60; the figure threshold starts at 0.25. Missing/ambiguous
figure or range observations hold motion. Calibration still uses the marker.
See `docs/FIGURE_MODEL.md` in the supplied C++ project for transfer/build commands.

References: [pinned Ultralytics exporter](https://github.com/ultralytics/ultralytics/blob/v8.3.228/ultralytics/engine/exporter.py),
[Coral compiler](https://coral.ai/docs/edgetpu/compiler/).